In [1]:
# =========================================================
# BƯỚC 1: CÀI ĐẶT & ÉP PYTHON NHẬN DIỆN THƯ MỤC
# =========================================================
import os
import sys
import json
import csv
from types import ModuleType

# Nếu cần dùng NMS
import torch
from torchvision.ops import nms

# Làm việc trong /kaggle/working
%cd /kaggle/working

# Clone DocLayout-YOLO nếu chưa có
if not os.path.exists("DocLayout-YOLO"):
    !git clone --depth 1 https://github.com/opendatalab/DocLayout-YOLO.git

%cd DocLayout-YOLO
!pip install -q -e .

# Ép repo vào sys.path
repo_dir = "/kaggle/working/DocLayout-YOLO"
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

%cd /kaggle/working

# =========================================================
# BƯỚC 2: "HACK" BỘ NHỚ ĐỂ FIX LỖI MODULE HUB
# =========================================================
if "doclayout_yolo.utils.callbacks.hub" not in sys.modules:
    dummy_hub = ModuleType("doclayout_yolo.utils.callbacks.hub")
    dummy_hub.callbacks = {}
    sys.modules["doclayout_yolo.utils.callbacks.hub"] = dummy_hub

/kaggle/working
Cloning into 'DocLayout-YOLO'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 277 (delta 41), reused 237 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 10.79 MiB | 25.33 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/DocLayout-YOLO
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doclayout_yolo (pyproject.toml) ... done
/kaggle/working


In [2]:
from pathlib import Path
import shutil
from doclayout_yolo import YOLOv10

print("✅ Import YOLOv10 thành công!")

# ---------------------------------------------------------
# HÀM HẬU XỬ LÝ BOX: NMS + PADDING
# ---------------------------------------------------------
def postprocess_boxes(results_list, # Sửa tham số này thành list
                      img_w: int,
                      img_h: int,
                      names,
                      iou_thr: float = 0.6,
                      pad_scale_x: float = 0.3,
                      pad_scale_y: float = 0.1,
                      agnostic_nms: bool = True):
    boxes = []
    scores = []
    labels = []

    # Duyệt qua các result từ cả 2 model
    for r in results_list:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])

            boxes.append([x1, y1, x2, y2])
            scores.append(conf)
            labels.append(cls_id)

    if len(boxes) == 0:
        return []

    boxes = torch.tensor(boxes)
    scores = torch.tensor(scores)
    labels = torch.tensor(labels)

    keep_indices = []

    if agnostic_nms:
        # NMS không phân biệt class (Class-Agnostic NMS)
        # Sẽ loại bỏ các box trùng lặp > iou_thr và ưu tiên giữ lại box có score cao nhất
        kept = nms(boxes, scores, iou_thr)
        keep_indices = kept.tolist()
    else:
        # NMS per-class (Chỉ xử lý trùng lặp trong cùng 1 class như code cũ)
        unique_classes = set(labels.tolist())
        for cls in unique_classes:
            cls_mask = labels == cls
            cls_indices = torch.nonzero(cls_mask, as_tuple=True)[0]
            cls_boxes = boxes[cls_indices]
            cls_scores = scores[cls_indices]

            kept = nms(cls_boxes, cls_scores, iou_thr)
            keep_indices.extend(cls_indices[kept].tolist())
            
        keep_indices = sorted(set(keep_indices))

    regions = []
    for idx in keep_indices:
        x1, y1, x2, y2 = boxes[idx].tolist()
        h = y2 - y1

        # Padding theo chiều cao dòng
        pad_x = pad_scale_x * h
        pad_y = pad_scale_y * h

        x1 = max(0.0, x1 - pad_x)
        y1 = max(0.0, y1 - pad_y)
        x2 = min(float(img_w), x2 + pad_x)
        y2 = min(float(img_h), y2 + pad_y)

        cls_id = int(labels[idx])
        cls_name = names[cls_id] if names is not None else "region"

        # Format đúng RUKOPYS/Kaggle: [x1, y1, x2, y2]
        regions.append({
            "bbox": [round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)],
            "type": cls_name,
            "text": ""  # tạm thời để rỗng, OCR sẽ fill sau
        })

    return regions

✅ Import YOLOv10 thành công!


In [3]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ---------------------------------------------------------

# CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
MODEL_PATH_1 = "/kaggle/input/datasets/notpitomon/htd-final-bbox-weight/Doclayout Yolo Final V 1.0.pt"
MODEL_PATH_2 = "/kaggle/input/datasets/notpitomon/htd-final-bbox-weight/Doclayout Yolo Final V 1.1.pt"

# ĐƯỜNG DẪN TẬP TEST CHÍNH THỨC CỦA COMPETITION
# Nếu đang debug với bản copy RUKOPYS riêng, bạn có thể đổi lại.
TEST_IMG_DIR = Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images")

image_paths = []
if TEST_IMG_DIR.exists():
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(TEST_IMG_DIR.glob(ext))
else:
    raise FileNotFoundError(f"Không tìm thấy thư mục: {TEST_IMG_DIR}")

image_paths = sorted(image_paths)

# Thư mục output
OUT_DIR = Path("/kaggle/working/inference_output")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# File metadata jsonl (debug) + file nộp Kaggle
META_PATH = OUT_DIR / "metadata.jsonl"
SUB_PATH = OUT_DIR / "submission.csv"

# Tham số YOLO
IMG_SIZE = 1280
CONF = 0.25   # Giữ conf thấp để không mất box
MAX_DET = 200

# Tham số hậu xử lý
IOU_NMS = 0.6
# PAD_SCALE_X = 0.3
# PAD_SCALE_Y = 0.1
PAD_SCALE_X = 0.0
PAD_SCALE_Y = 0.0

# assert Path(MODEL_PATH).exists(), f"Không tìm thấy model: {MODEL_PATH}"
# print(f"📸 Tìm thấy tổng cộng {len(image_paths)} ảnh cần dự đoán.")

# Khởi tạo model
print("\nĐang load model...")
model1 = YOLOv10(MODEL_PATH_1)
model2 = YOLOv10(MODEL_PATH_2)

print("🚀 Bắt đầu trích xuất tọa độ & sinh submission.csv...")

count_images = 0

with open(META_PATH, 'w', encoding='utf-8') as f_meta, \
     open(SUB_PATH, 'w', newline='', encoding='utf-8') as f_sub:

    writer = csv.writer(f_sub)
    writer.writerow(["image", "regions"])  # header đúng format Kaggle

    for img_path in image_paths:
        # Chạy dự đoán model 1
        results1 = model1.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=CONF,
            max_det=MAX_DET,
            verbose=False
        )
        
        # Chạy dự đoán model 2
        results2 = model2.predict(
            source=str(img_path),
            imgsz=IMG_SIZE,
            conf=CONF,
            max_det=MAX_DET,
            verbose=False
        )

        r1 = results1[0]
        r2 = results2[0]
        img_h, img_w = r1.orig_shape # Kích thước ảnh gốc của 2 kết quả là như nhau

        # Hậu xử lý: Truyền danh sách kết quả của cả 2 model vào
        regions = postprocess_boxes(
            [r1, r2], 
            img_w=img_w,
            img_h=img_h,
            names=model1.names, # Class names chung của 2 model
            iou_thr=IOU_NMS,
            pad_scale_x=PAD_SCALE_X,
            pad_scale_y=PAD_SCALE_Y
        )

        # Ghi metadata.jsonl để debug (giống schema RUKOPYS)
        data_record = {
            "file_name": f"images/{img_path.name}",
            "image_width": img_w,
            "image_height": img_h,
            "annotation_source": "yolov10_prediction",
            "regions": regions
        }
        f_meta.write(json.dumps(data_record, ensure_ascii=False) + "\n")

        # Ghi 1 dòng vào submission.csv
        # Lưu ý: cột "image" là tên file, "regions" là chuỗi JSON
        writer.writerow([img_path.name, json.dumps(regions, ensure_ascii=False)])

        count_images += 1
        if count_images % 50 == 0:
            print(f"⏳ Đã xử lý {count_images}/{len(image_paths)} ảnh...")

print("\n--- HOÀN TẤT ---")
print(f"Đã xử lý thành công: {count_images} ảnh")
print(f"File metadata: {META_PATH}")
print(f"File nộp Kaggle: {SUB_PATH}")


Đang load model...
🚀 Bắt đầu trích xuất tọa độ & sinh submission.csv...
⏳ Đã xử lý 50/385 ảnh...
⏳ Đã xử lý 100/385 ảnh...
⏳ Đã xử lý 150/385 ảnh...
⏳ Đã xử lý 200/385 ảnh...
⏳ Đã xử lý 250/385 ảnh...
⏳ Đã xử lý 300/385 ảnh...
⏳ Đã xử lý 350/385 ảnh...

--- HOÀN TẤT ---
Đã xử lý thành công: 385 ảnh
File metadata: /kaggle/working/inference_output/metadata.jsonl
File nộp Kaggle: /kaggle/working/inference_output/submission.csv


In [4]:
# # ---------------------------------------------------------
# # CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# # ---------------------------------------------------------
# from pathlib import Path
# import shutil
# import json
# import csv
# import cv2
# import matplotlib.pyplot as plt
# from doclayout_yolo import YOLOv10

# # Đường dẫn weights của bạn (đổi lại cho đúng)
# MODEL_PATH = "/kaggle/input/models/notpitomon/htd-box-doclayoutyolo-v4/pytorch/default/3/DoclayoutYoloV4.2.pt"

# # CHỈ ĐỊNH DUY NHẤT 2 ẢNH THEO YÊU CẦU
# image_paths = [
#     # Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/1dd10944-9049-4ca8-9944-fb83e3d1daeb.jpg"),
#     # Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/437d82cf-a0c3-4620-8211-90f673f51ed2.jpg"),
#     # Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/defa144a-9bd4-56f5-b951-c23607449449.jpg"),
#     # Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/24f7d274-b6ce-4495-939f-9528a1fe42f4.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/ab4a0f63-eb5a-5d60-bc2c-f3e482375cf1.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/052068c9-4ed7-5f3a-8951-c37a606a5af7.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/65fd757d-857d-53dc-8c51-f875a2553497.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/728d28d0-1c0a-53eb-8cd1-5d2123cf0a90.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/251ddee6-e8e2-58a3-91c2-eb4970a2648d.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/a6f4e8f6-f124-5d87-8bd6-c29267fa2a37.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/b5db1d37-867d-507e-8c9a-2b9d8667ce57.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/b67a07b4-da09-562d-9824-b886eb0b36fa.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/3e920349-b25e-5705-8d5c-839aed38f6c2.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/ce81b071-f963-558d-8c2b-ecf490f32658.jpg"),
#     Path("/kaggle/input/datasets/quii29/rukopys-dataset/test/images/011934b4-d919-5000-8c1e-39f96daeba68.jpg"),
# ]

# # Lọc các file tồn tại
# image_paths = [p for p in image_paths if p.exists()]

# # Thư mục output
# OUT_DIR = Path("/kaggle/working/inference_output")
# if OUT_DIR.exists():
#     shutil.rmtree(OUT_DIR)
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# # File metadata jsonl (debug) + file nộp Kaggle
# META_PATH = OUT_DIR / "metadata.jsonl"
# SUB_PATH = OUT_DIR / "submission.csv"

# # Tham số YOLO
# IMG_SIZE = 1280
# CONF = 0.1
# MAX_DET = 200

# # Tham số hậu xử lý
# IOU_NMS = 0.6
# PAD_SCALE_X = 0.0
# PAD_SCALE_Y = 0.0

# assert Path(MODEL_PATH).exists(), f"Không tìm thấy model: {MODEL_PATH}"
# print(f"📸 Tìm thấy tổng cộng {len(image_paths)} ảnh cần dự đoán.")

# # Khởi tạo model
# print("\nĐang load model...")
# model = YOLOv10(MODEL_PATH)

# print("🚀 Bắt đầu dự đoán & vẽ ảnh...")

# count_images = 0

# with open(META_PATH, 'w', encoding='utf-8') as f_meta, \
#      open(SUB_PATH, 'w', newline='', encoding='utf-8') as f_sub:

#     writer = csv.writer(f_sub)
#     writer.writerow(["image", "regions"])

#     for img_path in image_paths:
#         # Chạy dự đoán
#         results = model.predict(
#             source=str(img_path),
#             imgsz=IMG_SIZE,
#             conf=CONF,
#             max_det=MAX_DET,
#             verbose=False
#         )

#         r = results[0]
#         img_h, img_w = r.orig_shape

#         # Hậu xử lý: NMS + padding
#         regions = postprocess_boxes(
#             r,
#             img_w=img_w,
#             img_h=img_h,
#             names=model.names,
#             iou_thr=IOU_NMS,
#             pad_scale_x=PAD_SCALE_X,
#             pad_scale_y=PAD_SCALE_Y
#         )

#         # ---------------------------------------------------------
#         # VẼ BBOX VÀ CLASS LÊN ẢNH SAU ĐÓ HIỂN THỊ
#         # ---------------------------------------------------------
#         img_vis = cv2.imread(str(img_path))
#         img_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB)

#         for region in regions:
#             x1, y1, x2, y2 = map(int, map(round, region["bbox"]))
#             cls_name = region["type"]

#             # Vẽ bounding box màu đỏ (Red)
#             cv2.rectangle(img_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
#             # Ghi text (tên class) ngay trên góc trái của bbox
#             cv2.putText(
#                 img_vis, cls_name, (x1, max(y1 - 10, 0)), 
#                 cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 0), 3
#             )

#         # Hiển thị ảnh ngay trên notebook
#         plt.figure(figsize=(16, 16))
#         plt.imshow(img_vis)
#         plt.axis("off")
#         plt.title(f"Kết quả cho: {img_path.name}")
#         plt.show()
#         # ---------------------------------------------------------

#         # Ghi metadata.jsonl để debug
#         data_record = {
#             "file_name": f"images/{img_path.name}",
#             "image_width": img_w,
#             "image_height": img_h,
#             "annotation_source": "yolov10_prediction",
#             "regions": regions
#         }
#         f_meta.write(json.dumps(data_record, ensure_ascii=False) + "\n")

#         # Ghi 1 dòng vào submission.csv
#         writer.writerow([img_path.name, json.dumps(regions, ensure_ascii=False)])

#         count_images += 1

# print("\n--- HOÀN TẤT ---")
# print(f"Đã xử lý và vẽ thành công: {count_images} ảnh")
# print(f"File metadata: {META_PATH}")
# print(f"File nộp Kaggle: {SUB_PATH}")

In [5]:
import pandas as pd

df_sub = pd.read_csv(SUB_PATH)
display(df_sub.head())

print("\nSố dòng trong submission:", len(df_sub))
print("Cột:", list(df_sub.columns))

,image,regions
0,002b94ef-e000-4e76-bc7e-7846166bc806.jpg,"[{""bbox"": [702.46, 428.93, 3493.4, 686.89], ""t..."
1,00829c22-1e80-4e13-a395-3f9d8b088f7d.jpg,"[{""bbox"": [232.12, 2777.78, 2722.6, 2908.1], ""..."
2,00af1ec3-5315-4782-bc61-bf498617e5a2.jpg,"[{""bbox"": [324.0, 102.75, 890.49, 224.87], ""ty..."
3,010c6dad-2f5d-4e4c-802b-f978a7e1998b.jpg,"[{""bbox"": [722.96, 3679.21, 2816.97, 4041.24],..."
4,011934b4-d919-5000-8c1e-39f96daeba68.jpg,"[{""bbox"": [0.0, 2324.19, 251.59, 2474.29], ""ty..."



Số dòng trong submission: 385
Cột: ['image', 'regions']
